In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 24.1 MB/s eta 0:00:00


In [3]:
import optuna

In [ ]:
torch.cuda.is_available()

True

In [ ]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!ls "/content/drive/MyDrive/"

 Adobe_Photoshop_CC_2019_x64.rar
'Adobe Premiere Pro 2021 v15.0.0.41 (x64) Pre-Cracked {CracksHash}.rar'
'Colab Notebooks'
 dataset
 INFO
 yunish


In [8]:
df= pd.read_csv(r"/content/drive/MyDrive/dataset/fashion-mnist.csv")

In [9]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test= train_test_split(df.iloc[:,1: ], df['label'], random_state= 42, test_size= 0.2)

In [11]:
scaler= StandardScaler()

scaler.fit(x_train)

x_train= scaler.transform(x_train)
x_test= scaler.transform(x_test)

In [12]:
x_train= torch.tensor(x_train, dtype= torch.float32)
x_test= torch.tensor(x_test, dtype= torch.float32)
y_train= torch.tensor(y_train.values, dtype= torch.float32)
y_test= torch.tensor(y_test.values, dtype= torch.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomData(Dataset):
    def __init__ (self, features, labels):
        self.features= features
        self.label= labels


    def __len__(self):
        return self.features.shape[0]


    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [15]:
train_data= CustomData(x_train, y_train)

In [ ]:
test_data= CustomData(x_test, y_test)

In [ ]:
class MyNN(nn.Module):
    def __init__(self, input_features, output_features, num_hidden_layers, num_neurons, prob):

        super().__init__()

        layers= []

        for _ in range(num_hidden_layers):

            layers.append(nn.Linear(input_features, num_neurons))
            layers.append(nn.BatchNorm1d(num_neurons))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p= prob))

            input_features= num_neurons

        layers.append(nn.Linear(num_neurons, output_features))


        self.ann= nn.Sequential(*layers)


    def forward(self, data):
        return self.ann(data)



In [18]:
def objective(trial):
    input_features= x_train.shape[1]
    output_features= 10
    
    num_hidden_layers= trial.suggest_int("num_hidden_layers", 2, 8, step= 1)
    num_neurons= trial.suggest_int("num_neurons", 32, 128, step= 8)
    prob= trial.suggest_float("prob", 0.1, 0.5, step= 0.1)
    learnig_rate= trial.suggest_float('lr', 1e-5, 1e-1, log= True)
    opt= trial.suggest_categorical("optimizer", ['SGD', 'Adam', 'RmsProp'])
    wd= trial.suggest_float('weight_decay', 1e-5, 1e-1, log= True)
    batch_size= trial.suggest_categorical('batch_size', [8, 16, 32, 64])
    epoch= trial.suggest_int('epochs', 10, 100, step= 10)



    train_dataset= DataLoader(dataset= train_data, batch_size= batch_size, shuffle= True, pin_memory= True)
    test_dataset= DataLoader(dataset= test_data, batch_size= batch_size, shuffle= False, pin_memory= True)

    

    model= MyNN(input_features, output_features, num_hidden_layers, num_neurons, prob)
    model= model.to(device= device)

    if opt=='SGD':
        optimizer= torch.optim.SGD(params= model.parameters(), lr= learnig_rate, weight_decay= wd)

    if opt== "Adam":
        optimizer= torch.optim.Adam(params= model.parameters(),lr= learnig_rate, weight_decay= wd)

    if opt== 'RmsProp':
        optimizer = torch.optim.RMSprop(params= model.parameters(),lr= learnig_rate, weight_decay= wd)

    # defining loss function

    critetation= nn.CrossEntropyLoss()



    # training loop
    model.train()
    for _ in range(epoch):
        total_loss= 0

        for batch_features, batch_label in train_dataset:

            batch_features= batch_features.to(device= device)
            batch_label= batch_label.to(device= device)
            
            y_pred= model(batch_features)


            loss= critetation(y_pred, batch_label.to(dtype= torch.long))

            with torch.no_grad():
                total_loss+= loss

            #resets all the gradients
            optimizer.zero_grad()

            #calculate gradient of loss w.r.t each weight and biases
            loss.backward()



            #optimizes weights and biases
            optimizer.step()



    #Evaluation loop

    model.eval()

        # calculate accuracy
    total= 0
    correct = 0
    for test_batch, test_label in test_dataset:
        test_batch= test_batch.to(device= device)
        test_label= test_label.to(device= device)
        with torch.no_grad():
            y_pred= model(test_batch)

        _, idx= torch.max(y_pred, axis = 1)

        total= total + test_label.shape[0]

        correct= correct + (idx== test_label).sum()


    accuracy= correct/total

    return accuracy


            



In [19]:
study= optuna.create_study(direction= 'maximize', sampler= optuna.samplers.TPESampler())

study.optimize(func= objective, n_trials= 10)

[I 2026-09-02 04:56:01,909] A new study created in memory with name: no-name-e37d4e88-58a6-43d6-92aa-a87acc26660a
[I 2026-09-02 05:00:25,431] Trial 0 finished with value: 0.5635833144187927 and parameters: {'num_hidden_layers': 6, 'num_neurons': 48, 'prob': 0.5, 'lr': 0.0030190968152188705, 'optimizer': 'RmsProp', 'weight_decay': 0.0011257854283652081, 'batch_size': 16, 'epochs': 20}. Best is trial 0 with value: 0.5635833144187927.
[I 2026-09-02 05:04:00,146] Trial 1 finished with value: 0.8494166731834412 and parameters: {'num_hidden_layers': 3, 'num_neurons': 112, 'prob': 0.30000000000000004, 'lr': 0.0037458676861257443, 'optimizer': 'RmsProp', 'weight_decay': 0.0008073258307615273, 'batch_size': 64, 'epochs': 70}. Best is trial 1 with value: 0.8494166731834412.
[I 2026-09-02 05:14:18,331] Trial 2 finished with value: 0.7880833148956299 and parameters: {'num_hidden_layers': 5, 'num_neurons': 128, 'prob': 0.1, 'lr': 1.889342232584068e-05, 'optimizer': 'SGD', 'weight_decay': 3.31436241

In [20]:
study.best_value

0.8859166502952576

In [21]:
study.best_params

{'num_hidden_layers': 4,
 'num_neurons': 72,
 'prob': 0.30000000000000004,
 'lr': 5.7664923708465276e-05,
 'optimizer': 'Adam',
 'weight_decay': 0.00036894253462689,
 'batch_size': 64,
 'epochs': 80}

In [22]:
round(study.best_params['prob'], 1)

0.3